##** Setup**

In [0]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("/mnt/user-data/uploads")   # change if running locally, e.g. Path("./data")
OUT_DIR  = Path("./processed")
OUT_DIR.mkdir(exist_ok=True)

**Load raw data**

In [0]:
DATA_DIR = Path("/Volumes/project/default/project")

def read(name, **kw):
    return pd.read_csv(DATA_DIR / name, **kw)

orders = read("olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"])

items = read("olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])

payments = read("olist_order_payments_dataset.csv")

reviews = read("olist_order_reviews_dataset.csv",
               parse_dates=["review_creation_date", "review_answer_timestamp"])

customers = read("olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": "string"})
sellers   = read("olist_sellers_dataset.csv",   dtype={"seller_zip_code_prefix": "string"})
geo       = read("olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": "string"})
products  = read("olist_products_dataset.csv")
cat_tr    = read("product_category_name_translation.csv", encoding="utf-8-sig")

raw = dict(orders=orders, items=items, payments=payments, reviews=reviews,
           customers=customers, sellers=sellers, geo=geo, products=products, cat_tr=cat_tr)
pd.DataFrame({k: v.shape for k, v in raw.items()}, index=["rows", "cols"]).T

,rows,cols
orders,99441,8
items,112650,7
payments,103886,5
reviews,99224,7
customers,99441,5
sellers,3095,4
geo,1000163,5
products,32951,9
cat_tr,71,2


## **3. Profile raw data**

In [0]:
def profile(df, name):
    nulls = df.isna().sum()
    out = pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": nulls,
                        "null_%": (nulls / len(df) * 100).round(2), "n_unique": df.nunique()})
    print(f"\n=== {name}: {df.shape[0]:,} rows | {df.duplicated().sum():,} fully duplicated rows")
    return out[out.nulls > 0] if nulls.any() else out.head(0)

for k, v in raw.items():
    p = profile(v, k)
    if not p.empty:
        display(p)


=== orders: 99,441 rows | 0 fully duplicated rows


dtype,nulls,null_%,n_unique
datetime64[ns],160,0.16,90733
datetime64[ns],1783,1.79,81018
datetime64[ns],2965,2.98,95664



=== items: 112,650 rows | 0 fully duplicated rows

=== payments: 103,886 rows | 0 fully duplicated rows

=== reviews: 99,224 rows | 0 fully duplicated rows


dtype,nulls,null_%,n_unique
object,87656,88.34,4527
object,58247,58.7,36159



=== customers: 99,441 rows | 0 fully duplicated rows

=== sellers: 3,095 rows | 0 fully duplicated rows

=== geo: 1,000,163 rows | 261,831 fully duplicated rows

=== products: 32,951 rows | 0 fully duplicated rows


dtype,nulls,null_%,n_unique
object,610,1.85,73
float64,610,1.85,66
float64,610,1.85,2960
float64,610,1.85,19
float64,2,0.01,2204
float64,2,0.01,99
float64,2,0.01,102
float64,2,0.01,95



=== cat_tr: 71 rows | 0 fully duplicated rows


## 4. Clean each table

**Known issues in Olist and how each is handled**

| Table | Issue | Fix |
|---|---|---|
| geolocation | ~1M rows, many per zip prefix | Aggregate to one centroid per zip |
| products | Misspelt columns (`lenght`); ~600 missing categories; Portuguese names | Rename, fill `unknown`, join English translation |
| reviews | Duplicate `order_id`s; mostly empty comments | Keep latest review per order; add text flags |
| payments | Multiple rows per order (instalments, vouchers) | Aggregate to order level |
| orders | Null delivery dates for non-delivered orders; some timestamps out of order | Keep nulls, flag anomalies |
| text fields | Inconsistent case/whitespace in city names | `strip().lower()` |

In [0]:
# ---------- Geolocation: one row per zip prefix ----------
geo_zip = (geo.groupby("geolocation_zip_code_prefix", as_index=False)
              .agg(lat=("geolocation_lat", "mean"),
                   lng=("geolocation_lng", "mean"),
                   geo_state=("geolocation_state", lambda s: s.mode().iat[0])))
geo_zip = geo_zip.rename(columns={"geolocation_zip_code_prefix": "zip_prefix"})

# ---------- Customers / Sellers ----------
for df, col in [(customers, "customer_city"), (sellers, "seller_city")]:
    df[col] = df[col].str.strip().str.lower()

customers = customers.rename(columns={"customer_zip_code_prefix": "zip_prefix"})
sellers   = sellers.rename(columns={"seller_zip_code_prefix": "zip_prefix"})

customers = customers.merge(geo_zip[["zip_prefix", "lat", "lng"]], on="zip_prefix", how="left")
sellers   = sellers.merge(geo_zip[["zip_prefix", "lat", "lng"]], on="zip_prefix", how="left")

# ---------- Products ----------
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"})

products = products.merge(cat_tr, on="product_category_name", how="left")
products["category"] = (products["product_category_name_english"]
                        .fillna(products["product_category_name"])   # untranslated PT names kept
                        .fillna("unknown"))
products["product_volume_cm3"] = (products.product_length_cm *
                                  products.product_height_cm *
                                  products.product_width_cm)
products = products.drop(columns=["product_category_name", "product_category_name_english"])

# ---------- Reviews: one per order (latest wins) ----------
reviews = (reviews.sort_values("review_answer_timestamp")
                  .drop_duplicates("order_id", keep="last"))
reviews["has_comment"] = reviews["review_comment_message"].notna()
reviews["comment_length"] = reviews["review_comment_message"].fillna("").str.len()
reviews["is_negative"] = reviews["review_score"] <= 2
reviews = reviews[["order_id", "review_score", "has_comment", "comment_length", "is_negative"]]

print("customers", customers.shape, "| sellers", sellers.shape,
      "| products", products.shape, "| reviews", reviews.shape)

customers (99441, 7) | sellers (3095, 6) | products (32951, 10) | reviews (98673, 5)


## **5. Feature engineering: orders (delivery performance)**

In [0]:
o = orders.copy()

o["purchase_date"]  = o.order_purchase_timestamp.dt.normalize()
o["purchase_year"]  = o.order_purchase_timestamp.dt.year
o["purchase_month"] = o.order_purchase_timestamp.dt.to_period("M").astype(str)
o["purchase_dow"]   = o.order_purchase_timestamp.dt.day_name()
o["purchase_hour"]  = o.order_purchase_timestamp.dt.hour

hours = lambda a, b: (o[a] - o[b]).dt.total_seconds() / 3600
days  = lambda a, b: (o[a] - o[b]).dt.total_seconds() / 86400

o["approval_hours"]       = hours("order_approved_at", "order_purchase_timestamp")
o["carrier_handover_days"]= days("order_delivered_carrier_date", "order_purchase_timestamp")
o["delivery_days"]        = days("order_delivered_customer_date", "order_purchase_timestamp")
o["estimated_days"]       = days("order_estimated_delivery_date", "order_purchase_timestamp")
o["delay_days"]           = days("order_delivered_customer_date", "order_estimated_delivery_date")
o["is_late"]              = np.where(o.order_delivered_customer_date.notna(), o.delay_days > 0, np.nan)

# data-quality flags (kept, not dropped, so analysts can decide)
o["flag_delivered_no_date"] = (o.order_status == "delivered") & o.order_delivered_customer_date.isna()
o["flag_negative_approval"] = o.approval_hours < 0
o["flag_carrier_before_purchase"] = o.carrier_handover_days < 0

o[["delivery_days", "delay_days", "approval_hours"]].describe().T

,count,mean,std,min,25%,50%,75%,max
delivery_days,"96,476.00",12.56,9.55,0.53,6.77,10.22,15.72,209.63
delay_days,"96,476.00",-11.18,10.19,-146.02,-16.24,-11.95,-6.39,188.98
approval_hours,"99,281.00",10.42,26.04,0.00,0.21,0.34,14.58,"4,509.18"


## **6. Aggregate child tables to order grain**
Items and payments have many rows per order. Joining them directly to orders would *duplicate rows and inflate revenue*, so each is aggregated first.

In [0]:
# ---------- Items -> order level ----------
items_agg = (items.groupby("order_id")
    .agg(n_items=("order_item_id", "max"),
         n_distinct_products=("product_id", "nunique"),
         n_sellers=("seller_id", "nunique"),
         items_value=("price", "sum"),
         freight_value=("freight_value", "sum"))
    .reset_index())
items_agg["gmv"] = items_agg.items_value + items_agg.freight_value
items_agg["freight_share"] = items_agg.freight_value / items_agg.gmv

# ---------- Payments -> order level ----------
pay_agg = (payments.groupby("order_id")
    .agg(payment_value=("payment_value", "sum"),
         n_payment_methods=("payment_type", "nunique"),
         max_installments=("payment_installments", "max"))
    .reset_index())

# dominant payment type = type with the largest value in that order
main_type = (payments.sort_values("payment_value", ascending=False)
                     .drop_duplicates("order_id")[["order_id", "payment_type"]]
                     .rename(columns={"payment_type": "main_payment_type"}))
pay_agg = pay_agg.merge(main_type, on="order_id")

**## 7. Fact table A: `fact_orders` (one row per order)**

In [0]:
fact_orders = (o
    .merge(customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(pay_agg,   on="order_id", how="left")
    .merge(reviews,   on="order_id", how="left"))

fact_orders.shape

(99441, 40)

## **8. Fact table B: `fact_order_items` (one row per item, with product and seller attributes)**

In [0]:
fact_items = (items
    .merge(products[["product_id", "category", "product_weight_g", "product_volume_cm3",
                     "product_photos_qty"]], on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left")
    .merge(o[["order_id", "order_status", "order_purchase_timestamp", "purchase_month",
              "delivery_days", "delay_days", "is_late"]], on="order_id", how="left")
    .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]]
           .merge(o[["customer_id", "order_id"]], on="customer_id")[["order_id", "customer_unique_id", "customer_state"]],
           on="order_id", how="left"))

fact_items["item_total"] = fact_items.price + fact_items.freight_value
fact_items["same_state"] = fact_items.seller_state == fact_items.customer_state
fact_items.shape

(112650, 24)

##** 9. Customer table with RFM**
`customer_id` is *per order*; the real customer key is `customer_unique_id`. Repeat-purchase analysis must group on the latter.

In [0]:
delivered = fact_orders[fact_orders.order_status == "delivered"].copy()
snapshot = delivered.order_purchase_timestamp.max() + pd.Timedelta(days=1)

rfm = (delivered.groupby("customer_unique_id")
    .agg(first_order=("order_purchase_timestamp", "min"),
         last_order=("order_purchase_timestamp", "max"),
         frequency=("order_id", "nunique"),
         monetary=("payment_value", "sum"),
         avg_review=("review_score", "mean"),
         state=("customer_state", "last"))
    .reset_index())
rfm["recency_days"] = (snapshot - rfm.last_order).dt.days
rfm["is_repeat"] = rfm.frequency > 1

# quintile scores (rank first, because most customers have frequency = 1 and qcut would fail on ties)
rfm["R"] = pd.qcut(rfm.recency_days.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm.frequency.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm.monetary.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["rfm_score"] = rfm.R.astype(str) + rfm.F.astype(str) + rfm.M.astype(str)

def segment(r):
    if r.R >= 4 and r.F >= 4: return "Champions"
    if r.R >= 4 and r.M >= 4: return "Big spenders (recent)"
    if r.R >= 4:              return "Recent / new"
    if r.R <= 2 and r.M >= 4: return "At risk (high value)"
    if r.R <= 2:              return "Lapsed"
    return "Needs attention"
rfm["segment"] = rfm.apply(segment, axis=1)

print(f"Repeat customer rate: {rfm.is_repeat.mean():.1%}")
rfm.segment.value_counts()

Repeat customer rate: 3.0%


segment
Lapsed                   22808
Needs attention          18672
Champions                14871
At risk (high value)     14535
Recent / new             13529
Big spenders (recent)     8943
Name: count, dtype: int64

##** 10. Seller scorecard**

In [0]:
seller_perf = (fact_items[fact_items.order_status == "delivered"]
    .groupby("seller_id")
    .agg(orders=("order_id", "nunique"),
         items_sold=("order_item_id", "count"),
         revenue=("price", "sum"),
         avg_price=("price", "mean"),
         avg_review=("review_score", "mean"),
         late_rate=("is_late", "mean"),
         avg_delivery_days=("delivery_days", "mean"),
         n_categories=("category", "nunique"))
    .reset_index()
    .merge(sellers[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left"))

seller_perf.sort_values("revenue", ascending=False).head(10)

,seller_id,orders,items_sold,revenue,avg_price,avg_review,late_rate,avg_delivery_days,n_categories,seller_city,seller_state
834,4869f7a5dfa277a7dca6462dcf3b52b2,1124,1148,"226,987.93",197.72,4.14,0.12,15.01,10,guariba,SP
982,53243585a1d6dc2643021fd1853d8905,348,400,"217,940.44",544.85,4.13,0.04,13.37,2,lauro de freitas,BA
858,4a3ca9315b744ce9f8e9374361493884,1772,1949,"196,882.12",101.02,3.83,0.11,14.42,7,ibitinga,SP
2903,fa1c13f2614d7b5c4749cbc52fecda94,578,579,"190,917.14",329.74,4.37,0.10,13.34,5,sumare,SP
1480,7c67e1448b00f6e969d365cea6b010ab,973,1355,"186,570.05",137.69,3.35,0.10,22.39,6,itaquaquecetuba,SP
1504,7e93a43ef30c4f03f38b393420bc753a,319,322,"165,981.49",515.47,4.36,0.06,11.34,7,barueri,SP
2543,da8622b14eb17ae2831f4ac5b9dab84a,1311,1548,"159,816.87",103.24,4.07,0.07,11.17,4,piracicaba,SP
1450,7a67c85e85bb2ce8582c35f2203ad736,1145,1155,"139,658.69",120.92,4.27,0.06,11.17,2,sao paulo,SP
188,1025f0e2d44d7041d6cf58b6550e0bfa,910,1420,"138,208.56",97.33,3.86,0.09,12.05,4,sao paulo,SP
1758,955fee9216a65b617aa5c0531780ce60,1261,1472,"131,836.71",89.56,4.09,0.08,10.74,23,sao paulo,SP


## 11. Validation checks
These guard against the most common silent failure in pipelines: join fan-out that duplicates rows and inflates revenue.

In [0]:
checks = {}

# 1. grain: order-level table has exactly one row per order
checks["fact_orders row count == raw orders"] = len(fact_orders) == len(orders)
checks["fact_orders.order_id unique"] = fact_orders.order_id.is_unique

# 2. item-level table keeps every raw item row
checks["fact_items row count == raw items"] = len(fact_items) == len(items)

# 3. revenue reconciliation
checks["item price total reconciles"] = np.isclose(fact_items.price.sum(), items.price.sum())
checks["freight total reconciles"]    = np.isclose(fact_items.freight_value.sum(), items.freight_value.sum())
checks["payment total reconciles"]    = np.isclose(fact_orders.payment_value.sum(), payments.payment_value.sum())

# 4. key integrity
checks["no orphan customers"] = fact_orders.customer_unique_id.notna().all()
checks["no negative prices"]  = (items.price >= 0).all()
checks["RFM keys unique"]     = rfm.customer_unique_id.is_unique

result = pd.Series(checks, name="passed")
display(result.to_frame())
assert result.all(), "Validation failed: investigate before exporting"

print("\nData-quality flags (informational):")
fact_orders[["flag_delivered_no_date", "flag_negative_approval", "flag_carrier_before_purchase"]].sum()

passed
true
true
true
true
true
true
true
true
true



Data-quality flags (informational):


flag_delivered_no_date            8
flag_negative_approval            0
flag_carrier_before_purchase    166
dtype: int64

## 12. Quick sanity analytics

In [0]:
monthly = (fact_orders[fact_orders.order_status == "delivered"]
           .groupby("purchase_month")
           .agg(orders=("order_id", "count"), gmv=("gmv", "sum"),
                avg_review=("review_score", "mean"), late_rate=("is_late", "mean"))
           .reset_index())
monthly.tail(6)

,purchase_month,orders,gmv,avg_review,late_rate
17,2018-03,7003,"1,120,598.24",3.81,0.21
18,2018-04,6798,"1,132,878.93",4.21,0.05
19,2018-05,6749,"1,128,774.52",4.24,0.08
20,2018-06,6099,"1,011,978.29",4.31,0.01
21,2018-07,6159,"1,027,807.28",4.32,0.04
22,2018-08,6351,"985,491.64",4.31,0.10


In [0]:
top_cat = (fact_items[fact_items.order_status == "delivered"]
           .groupby("category")
           .agg(revenue=("price", "sum"), items=("order_id", "count"), avg_review=("review_score", "mean"))
           .sort_values("revenue", ascending=False).head(10))
top_cat

,revenue,items,avg_review
category,,,
health_beauty,"1,233,131.72",9465,4.19
watches_gifts,"1,166,176.98",5859,4.07
bed_bath_table,"1,023,434.76",10953,3.92
sports_leisure,"954,852.55",8431,4.17
computers_accessories,"888,724.61",7644,3.99
furniture_decor,"711,927.69",8160,3.95
housewares,"615,628.69",6795,4.11
cool_stuff,"610,204.10",3718,4.20
auto,"578,966.65",4140,4.12


## 13. Export

In [0]:
tables = {"fact_orders": fact_orders, "fact_order_items": fact_items,
          "dim_customer_rfm": rfm, "dim_seller_performance": seller_perf,
          "dim_products": products, "dim_geo_zip": geo_zip}

try:
    import pyarrow  # noqa: F401
    for name, df in tables.items():
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)
    print("Saved Parquet files to", OUT_DIR.resolve())
except ImportError:
    for name, df in tables.items():
        df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print("pyarrow not installed; saved CSVs to", OUT_DIR.resolve())

Saved Parquet files to /Workspace/Users/mahima05bablr@xime.org/HandsOnBigData/processed
